In [6]:
import pybryt
import nbformat

In [7]:
# ── PyBryt 0.7.0 Windows compatibility patch ──────────────────────────────────
#
# THREE BUGS FIXED
# ─────────────────
# Bug 1 — EOFError: mkstemp() on Windows returns a path with backslashes
#   (e.g. C:\Users\Temp\tmpXXXXXX). When embedded raw into a generated code
#   cell string, backslashes become Python escape sequences, corrupting the path.
#   The subprocess cell crashes silently, the footprint file is never written,
#   and dill.load() reads an empty file → EOFError.
#   Fix: replace backslashes with forward slashes before embedding.
#
# Bug 2 — PermissionError on os.remove(): mkstemp() returns (fd, path) where
#   fd is an open OS-level file descriptor. Windows locks files with open
#   handles, so the subprocess cannot write to it AND os.remove() fails after
#   reading. Fix: call os.close(fd) immediately after mkstemp().
#
# Bug 3 — Wrong patch target: pybryt.student._execute() calls execute_notebook
#   via a locally-bound name created by `from .execution import execute_notebook`.
#   Patching pybryt.execution.execute_notebook has no effect. Must patch
#   pybryt.student.execute_notebook instead.
#
# Bug 4 — si.steps does not exist in 0.7.0. Use si.footprint.num_steps.
# ──────────────────────────────────────────────────────────────────────────────

import pybryt.student as _student
from pybryt.preprocessors import NotebookPreprocessor
from pybryt.utils import make_secret
import nbformat as _nbformat
import os as _os, dill as _dill
from copy import deepcopy as _deepcopy
from tempfile import mkstemp as _mkstemp
from textwrap import dedent as _dedent
from nbconvert.preprocessors import ExecutePreprocessor as _EP

def _patched_execute_notebook(nb, nb_path, addl_filenames=[], timeout=1200):
    nb = _deepcopy(nb)
    preprocessor = NotebookPreprocessor()
    nb = preprocessor.preprocess(nb)

    # Bug 2 fix: close the fd immediately so Windows releases the file lock
    fd, footprint_fp = _mkstemp()
    _os.close(fd)

    # Bug 1 fix: forward slashes survive embedding into Python source strings
    safe_fp = footprint_fp.replace("\\", "/")

    nb_dir = _os.path.abspath(_os.path.split(nb_path)[0])
    secret = make_secret()
    ftv = f"frame_tracer_{secret}"

    first_cell = _nbformat.v4.new_code_cell(_dedent(f"""
        import inspect, sys
        from pybryt.execution import FrameTracer
        {ftv} = FrameTracer(inspect.currentframe())
        {ftv}.start_trace(addl_filenames={addl_filenames})
        %cd {nb_dir}
    """))

    last_cell = _nbformat.v4.new_code_cell(_dedent(f"""
        {ftv}.end_trace()
        footprint = {ftv}.get_footprint()
        footprint.filter_out_unpickleable_values()
        import dill
        with open("{safe_fp}", "wb+") as f:
            dill.dump(footprint, f)
    """))

    nb["cells"].insert(0, first_cell)
    nb["cells"].append(last_cell)
    _EP(timeout=timeout, allow_errors=True).preprocess(nb)

    with open(footprint_fp, "rb") as f:
        footprint = _dill.load(f)
    _os.remove(footprint_fp)
    footprint.add_imports(*preprocessor.get_imports())
    footprint.set_executed_notebook(nb)
    return footprint

# Bug 3 fix: patch the name in pybryt.student's namespace, not pybryt.execution
_student.execute_notebook = _patched_execute_notebook

In [8]:
ref = pybryt.ReferenceImplementation.compile("lv2-3-2 parse-html.ipynb") #需要根據設計關卡 手動更改
ref.dump("../references/chapter-2-level-3-2.pkl") 

In [9]:
with open("lv2-3-2 parse-html.ipynb", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

ref = pybryt.ReferenceImplementation.compile(
    nb,
    name="chapter-2-level-3-2"
)

ref.dump("../references/chapter-2-level-3-2.pkl")


In [10]:
ref = pybryt.ReferenceImplementation.load("../references/chapter-2-level-3-2.pkl") #使用相對路徑
ref
